In [0]:
-- DROP TABLE IF EXISTS aviation.dev.gold_flight_revenue;

CREATE TABLE IF NOT EXISTS aviation.dev.gold_flight_revenue (
    flight_id bigint,
    booking_date date,
    sum_base_fare_try double,
    sum_taxes_try double,
    sum_total_fare double
)
CLUSTER BY (flight_id, booking_date);

In [0]:
MERGE INTO aviation.dev.gold_flight_revenue t
USING (
    SELECT 
        flight_id, 
        booking_date, 
        coalesce(sum(base_fare_try), 0) as sum_base_fare_try,
        coalesce(sum(taxes_try), 0) as sum_taxes_try,
        coalesce(sum(total_fare), 0) as sum_total_fare
    FROM aviation.dev.silver_bookings
    WHERE booking_date > '2026-10-01'
    GROUP BY flight_id, booking_date
) s
ON t.flight_id = s.flight_id AND t.booking_date = s.booking_date
WHEN MATCHED AND
    t.sum_base_fare_try <> s.sum_base_fare_try OR
    t.sum_taxes_try <> s.sum_taxes_try OR
    t.sum_total_fare <> s.sum_total_fare
    THEN UPDATE SET
        t.sum_base_fare_try = s.sum_base_fare_try,
        t.sum_taxes_try = s.sum_taxes_try,
        t.sum_total_fare = s.sum_total_fare
WHEN NOT MATCHED
    THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
-- DROP TABLE IF EXISTS aviation.dev.gold_customer_summary;

CREATE TABLE IF NOT EXISTS aviation.dev.gold_customer_summary (
    customer_id bigint,
    total_spend double,
    total_flights int,
    tier string
)
CLUSTER BY (customer_id);

In [0]:
MERGE INTO aviation.dev.gold_customer_summary t
USING (
    SELECT 
        customer_id, 
        loyalty_tier,
        sum(total_fare) as total_spend,
        count(*) as total_flights
    FROM aviation.dev.silver_bookings
    GROUP BY customer_id, loyalty_tier
) s
ON t.customer_id = s.customer_id
WHEN MATCHED AND
    t.total_spend <> s.total_spend OR
    t.total_flights <> s.total_flights OR
    t.tier <> s.loyalty_tier
    THEN UPDATE SET 
        t.total_spend = s.total_spend,
        t.total_flights = s.total_flights,
        t.tier = s.loyalty_tier
WHEN NOT MATCHED
    THEN INSERT (customer_id, total_spend, total_flights, tier)
    VALUES (s.customer_id, s.total_spend, s.total_flights, s.loyalty_tier)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
-- SELECT * FROM aviation.dev.gold_flight_revenue

-- DESCRIBE DETAIL aviation.dev.gold_flight_revenue



-- OPTIMIZE aviation.dev.gold_flight_revenue

-- VACUUM aviation.dev.gold_flight_revenue

-- DESCRIBE DETAIL aviation.dev.gold_flight_revenue

/*
the table is
      1 file / ~29 KB, so OPTIMIZE had nothing to compact
      and numFiles/sizeInBytes were unchanged; VACUUM with
      default 7-day retention removed nothing because every
      version is minutes old.
*/

-- SELECT * FROM aviation.dev.gold_customer_summary

Predictive Optimization: It is an autonomous optimization handled by Databricks. You should alter catalog and schemas to enable it.

example:

ALTER SCHEMA aviation.dev ENABLE PREDICTIVE OPTIMIZATION;
-- ya da tüm catalog için:
ALTER CATALOG aviation ENABLE PREDICTIVE OPTIMIZATION;